In [17]:
include("./helper_functions.jl")
include("./q_properties.jl")
include("./q_matroids.jl")
#include("./enumeration.jl")
#include("./optimizied_enumeration.jl")
#include("./database.jl")
using DataFrames
using SQLite
#using Posets

# Compute the rank polytope
***
*** 

In [5]:
RP, order, A, b = Rank_polytope(2,4)
#println(A)

(A polyhedron in ambient dimension 66, Vector{Any}[[1, [1 0 0 0]], [2, [0 1 0 0]], [3, [1 1 0 0]], [4, [0 0 1 0]], [5, [1 0 1 0]], [6, [0 1 1 0]], [7, [1 1 1 0]], [8, [0 0 0 1]], [9, [1 0 0 1]], [10, [0 1 0 1]]  …  [57, [1 0 0 0; 0 1 0 1; 0 0 1 1]], [58, [0 1 0 0; 0 0 1 0; 0 0 0 1]], [59, [1 0 0 1; 0 1 0 0; 0 0 1 0]], [60, [1 0 1 0; 0 1 0 0; 0 0 0 1]], [61, [1 0 0 1; 0 1 0 0; 0 0 1 1]], [62, [1 1 0 0; 0 0 1 0; 0 0 0 1]], [63, [1 0 0 1; 0 1 0 1; 0 0 1 0]], [64, [1 0 1 0; 0 1 1 0; 0 0 0 1]], [65, [1 0 0 1; 0 1 0 1; 0 0 1 1]], [66, [1 0 0 0; 0 1 0 0; 0 0 1 0; 0 0 0 1]]], [-1 0 … 0 0; 0 -1 … 0 0; … ; 0 0 … -1 1; 0 0 … -1 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [11]:
v = ones(Int,66)
#v in RP

66-element Vector{Int64}:
 1
 1
 1
 1
 1
 1
 1
 1
 1
 1
 ⋮
 1
 1
 1
 1
 1
 1
 1
 1
 1

In [6]:
V = collect(Oscar.vertices(RP))

11-element Vector{PointVector{fmpq}}:
 [0, 0, 0, 0, 0]
 [1, 1, 1, 1, 2]
 [0, 1, 1, 1, 1]
 [1, 0, 1, 1, 1]
 [1, 1, 0, 1, 1]
 [1, 1//2, 1//2, 1//2, 1]
 [1, 1, 1, 1, 1]
 [1, 1, 1, 0, 1]
 [1//2, 1, 1//2, 1//2, 1]
 [1//2, 1//2, 1, 1//2, 1]
 [1//2, 1//2, 1//2, 1, 1]

In [11]:
dim(RP)

5

In [4]:
f_vector(RP)

4-element Vector{fmpz}:
 6
 15
 18
 9

In [10]:
lattice_points(RP)

7-element SubObjectIterator{PointVector{fmpz}}:
 [0, 0, 0, 0, 0]
 [0, 1, 1, 1, 1]
 [1, 0, 1, 1, 1]
 [1, 1, 0, 1, 1]
 [1, 1, 1, 0, 1]
 [1, 1, 1, 1, 1]
 [1, 1, 1, 1, 2]

In [12]:
interior_lattice_points(RP)

0-element SubObjectIterator{PointVector{fmpz}}

In [4]:
function check_duality(Vert,order)
    is_true = true
    
    for V in Vert
        dual_vert = []
        for (id,x) in enumerate(V)
            if id == length(V)
                dual_value = rank(order[id][2])-V[length(V)]
                push!(dual_vert,dual_value)
            else
                ortho_space = orthogonal_complementV2(order[id][2])[1]
                id_ortho_space = [y[1] for y in order if y[2]==ortho_space][1]
                dual_value = rank(order[id][2])-V[length(V)]+V[id_ortho_space]
                push!(dual_vert,dual_value)
            end
        end
        #println(V)
        #println(dual_vert)

        if !(dual_vert in Vert)
            fail = V
            is_true = false
            break
        end
    end

    if is_true
        return is_true
    else
        return is_true, fail
    end
    
end

check_duality (generic function with 1 method)

In [5]:
@time check_duality(V,order)

  0.139277 seconds (239.79 k allocations: 15.927 MiB, 99.50% compilation time)


true

# Constructing Q-Matroid from the struct
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
A = matrix(Ext_F,[1 0 0;0 1 0])
QM1 = q_matroid_from_matrix(A)

Q-Matroid of rank 2 in 3-dim. vector-space over the Galois field with characteristic 2

In [9]:
bases = QM1.bases

4-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 1 1]
 [1 0 1; 0 1 0]
 [1 0 1; 0 1 1]

In [10]:
id_mat = matrix(GF(3),[1 0 0;0 1 0;0 0 1])
QM2 = Q_Matroid(id_mat,bases)

Q-Matroid of rank 2 in 3-dim. vector-space over the Galois field with characteristic 2

In [11]:
QM2.bases

4-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 1 1]
 [1 0 1; 0 1 0]
 [1 0 1; 0 1 1]

# Constructing the Uniform Q-Matroid
***
***

In [14]:
UQM = Uniform_q_matroid(GF(2),2,3)

Q-Matroid of rank 2 in 3-dim. vector-space over the Galois field with characteristic 2

In [15]:
UQM.bases

7-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]

# Constructing Q-Matroid from the Independentspaces
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
A = matrix(Ext_F,[1 0 0])
QM1 = q_matroid_from_matrix(A)
QM1.bases

4-element Vector{Any}:
 [1   0   0]
 [1   1   0]
 [1   0   1]
 [1   1   1]

In [25]:
indeps = Q_Matroid_Independentspaces(QM1)

5-element Vector{Any}:
 [0   0   0]
 [1   0   0]
 [1   1   0]
 [1   0   1]
 [1   1   1]

In [26]:
QM2 = q_matroid_from_independentspaces(indeps)

Q-Matroid of rank 1 in 3-dim. vector-space over the Galois field with characteristic 2

In [27]:
QM2.bases

4-element Vector{Any}:
 [1   0   0]
 [1   1   0]
 [1   0   1]
 [1   1   1]

# Constructing Q-Matroid from the Dependentspaces
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
A = matrix(Ext_F,[1 0 x])
QM1 = q_matroid_from_matrix(A)
QM1.bases

6-element Vector{Any}:
 [1   0   0]
 [1   1   0]
 [0   0   1]
 [1   0   1]
 [0   1   1]
 [1   1   1]

In [29]:
deps = Q_Matroid_Dependentspaces(QM1)

9-element Vector{Any}:
 [0   1   0]
 [1 0 0; 0 1 0]
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]
 [1 0 0; 0 1 0; 0 0 1]

In [30]:
QM2 = q_matroid_from_dependentspaces(deps)

Q-Matroid of rank 1 in 3-dim. vector-space over the Galois field with characteristic 2

In [31]:
QM2.bases

6-element Vector{Any}:
 [1   0   0]
 [1   1   0]
 [0   0   1]
 [1   0   1]
 [0   1   1]
 [1   1   1]

# Constructing Q-Matroid from the Circuits
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
A = matrix(Ext_F,[1 0 0;0 1 0])
QM1 = q_matroid_from_matrix(A)
QM1.bases

4-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 1 1]
 [1 0 1; 0 1 0]
 [1 0 1; 0 1 1]

In [35]:
circs = Q_Matroid_Circuits(QM1)

1-element Vector{Any}:
 [0   0   1]

In [36]:
QM2 = q_matroid_from_circuits(circs)

Q-Matroid of rank 2 in 3-dim. vector-space over the Galois field with characteristic 2

In [37]:
QM2.bases

4-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 1 1]
 [1 0 1; 0 1 0]
 [1 0 1; 0 1 1]

# Constructing Q-Matroid from the Hyperplanes
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
A = matrix(Ext_F,[1 0 0;0 1 x])
QM1 = q_matroid_from_matrix(A)
QM1.bases

6-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]

In [44]:
hyperps = Q_Matroid_Hyperplanes(QM1)

5-element Vector{Any}:
 [1   0   0]
 [1   1   0]
 [1   0   1]
 [1   1   1]
 [0 1 0; 0 0 1]

In [45]:
QM2 = q_matroid_from_hyperplanes(hyperps)

Q-Matroid of rank 2 in 3-dim. vector-space over the Galois field with characteristic 2

In [46]:
QM2.bases

6-element Vector{Any}:
 [1 0 0; 0 0 1]
 [1 1 0; 0 0 1]
 [1 0 0; 0 1 0]
 [1 0 1; 0 1 0]
 [1 0 0; 0 1 1]
 [1 0 1; 0 1 1]

In [47]:
Set(QM1.bases) == Set(QM2.bases) 

true

# Helper_function concerning subspaces
***
***

## All subs of fixed dim

In [48]:
q_binomcoeff(2,4,2)

35

In [49]:
test = subspaces_fix_dim(GF(2),0,4)
test[1]

[0   0   0   0]

In [ ]:
ms = matrix_space(GF(2),2,2)
zero_vec = matrix(GF(2),zeros(Int,1,2))
zero_vec

[0   0]

## All subspaces

In [53]:
test = all_subspaces(GF(2),3)
test

16-element Vector{Any}:
 [0   0   0]
 [1   0   0]
 [0   1   0]
 [1   1   0]
 [0   0   1]
 [1   0   1]
 [0   1   1]
 [1   1   1]
 [1 0 0; 0 1 0]
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]
 [1 0 0; 0 1 0; 0 0 1]

## Dimension of a fixed spaces

In [ ]:
@time subspace_dim(GF(2),matrix(GF(2),[0 1 0]))

## Set of elements of a fixed space

In [54]:
A = matrix(GF(2),[1 0 0 0 0;0 1 0 0 0;0 0 1 0 0;0 0 0 1 0;0 0 0 0 1])
B = matrix(GF(2),[1 0 0 0;0 1 0 0;0 0 1 0]);
C = matrix(GF(2),[1 0 0;0 1 0])
# typeof(sum_vs(GF(2),A,B));

[1   0   0]
[0   1   0]

In [55]:
subspace_set(B)

Set{Any} with 8 elements:
  [0 1 1 0]
  [0 0 1 0]
  [1 1 1 0]
  [1 0 1 0]
  [0 1 0 0]
  [0 0 0 0]
  [1 1 0 0]
  [1 0 0 0]

## Subspaces of a fixed space 

In [56]:
A = matrix(GF(2),[1 0 0 0 0;0 1 0 0 0;0 0 1 0 0;0 0 0 1 0;0 0 0 0 1])
B = matrix(GF(3),[1 0 0 0;0 1 0 0;0 0 1 0;0 0 0 1]);
C = matrix(GF(2),[1 0 0;0 1 0]);
# typeof(sum_vs(GF(2),A,B));

[1   0   0]
[0   1   0]

In [57]:
spaces = subspaces_fix_space(C)

3-element Vector{AbstractVector{Any}}:
 [[0 0 0]]
 [[1 1 0], [0 1 0], [1 0 0]]
 [[1 0 0; 0 1 0]]

In [ ]:
x = 0
for list in spaces
    x += length(list)
end
x

## All spaces that contain a fixed space

In [58]:
A = matrix(GF(2),[1 0 0 0 0;0 1 0 0 0;0 0 1 0 0;0 0 0 1 0;0 0 0 0 1])
B = matrix(GF(2),[1 0 0 0]);
C = matrix(GF(2),[1 0 0; 0 1 0; 0 0 1])
# typeof(sum_vs(GF(2),A,B));

[1   0   0]
[0   1   0]
[0   0   1]

In [60]:
@time containments = containments_fix_space(B)

  0.001340 seconds (20.18 k allocations: 1.027 MiB)


16-element Vector{Any}:
 [1   0   0   0]
 [1 0 0 0; 0 1 0 0]
 [1 0 0 0; 0 0 1 0]
 [1 0 0 0; 0 1 1 0]
 [1 0 0 0; 0 0 0 1]
 [1 0 0 0; 0 1 0 1]
 [1 0 0 0; 0 0 1 1]
 [1 0 0 0; 0 1 1 1]
 [1 0 0 0; 0 1 0 0; 0 0 1 0]
 [1 0 0 0; 0 1 0 0; 0 0 0 1]
 [1 0 0 0; 0 1 0 0; 0 0 1 1]
 [1 0 0 0; 0 0 1 0; 0 0 0 1]
 [1 0 0 0; 0 1 0 1; 0 0 1 0]
 [1 0 0 0; 0 1 1 0; 0 0 0 1]
 [1 0 0 0; 0 1 0 1; 0 0 1 1]
 [1 0 0 0; 0 1 0 0; 0 0 1 0; 0 0 0 1]

In [62]:
@time containments_fix_spaceV2(B)

  0.039572 seconds (67.36 k allocations: 4.648 MiB, 97.09% compilation time)


16-element Vector{Any}:
 [1   0   0   0]
 [1 0 0 0; 0 1 0 0]
 [1 0 0 0; 0 0 1 0]
 [1 0 0 0; 0 1 1 0]
 [1 0 0 0; 0 0 0 1]
 [1 0 0 0; 0 1 0 1]
 [1 0 0 0; 0 0 1 1]
 [1 0 0 0; 0 1 1 1]
 [1 0 0 0; 0 1 0 0; 0 0 1 0]
 [1 0 0 0; 0 1 0 0; 0 0 0 1]
 [1 0 0 0; 0 1 0 0; 0 0 1 1]
 [1 0 0 0; 0 0 1 0; 0 0 0 1]
 [1 0 0 0; 0 1 0 1; 0 0 1 0]
 [1 0 0 0; 0 1 1 0; 0 0 0 1]
 [1 0 0 0; 0 1 0 1; 0 0 1 1]
 [1 0 0 0; 0 1 0 0; 0 0 1 0; 0 0 0 1]

## Create all possible matrices of the field

In [ ]:
Ext_F,x = finite_field(2,1,"x")
matrix_collec(Ext_F,2,3)

4-element Vector{Any}:
 [1 0 0; 0 1 1]
 [1 0 0; 0 1 0]
 [1 0 1; 0 1 0]
 [1 0 1; 0 1 1]

## Create all possible gln-matrices of the field

In [66]:
gln_matrices(GF(2),2)

6-element Vector{Any}:
 [1 0; 0 1]
 [1 0; 1 1]
 [0 1; 1 0]
 [0 1; 1 1]
 [1 1; 1 0]
 [1 1; 0 1]

## Compute Möbius-function of subspaces

In [67]:
A = matrix(GF(2),[0 0 0])
B = matrix(GF(2),[1 0 0; 0 1 0]);

In [68]:
Möbius_func_subspace_lat(A,B)

2

## Compute codim one and dim one subspaces of a space

In [69]:
A = matrix(GF(2),[1 0 0;0 1 0;0 0 1])

[1   0   0]
[0   1   0]
[0   0   1]

In [70]:
codim_one_subs(A)

7-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 1; 0 1 1]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [0 1 0; 0 0 1]
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]

In [71]:
dim_one_subs(A)

7-element Vector{Any}:
 [1   1   0]
 [0   1   0]
 [1   0   1]
 [0   0   1]
 [1   0   0]
 [1   1   1]
 [0   1   1]

## Embedding spaces in the standard way

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 x;0 1 0])
QM = q_matroid_from_matrix(Mat)

Q-Matroid of rank 2 in 3-dim. vector-space over the Galois field with characteristic 2

In [74]:
indeps = Q_Matroid_Independentspaces(QM)

14-element Vector{Any}:
 [0   0   0]
 [1   1   0]
 [0   1   0]
 [1   0   0]
 [1   1   1]
 [0   1   1]
 [0   0   1]
 [1   0   1]
 [1 0 0; 0 1 0]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]

In [75]:
@time standard_embedding_higher_dim(indeps,1)

  0.045646 seconds (29.50 k allocations: 2.018 MiB, 99.63% compilation time)


14-element Vector{Any}:
 [0   0   0   0]
 [0   1   1   0]
 [0   0   1   0]
 [0   1   0   0]
 [0   1   1   1]
 [0   0   1   1]
 [0   0   0   1]
 [0   1   0   1]
 [0 1 0 0; 0 0 1 0]
 [0 1 0 0; 0 0 1 1]
 [0 0 1 0; 0 0 0 1]
 [0 1 0 1; 0 0 1 0]
 [0 1 1 0; 0 0 0 1]
 [0 1 0 1; 0 0 1 1]

In [76]:
@time standard_embedding_higher_dimV2(indeps,1)

  0.150055 seconds (285.88 k allocations: 15.953 MiB, 99.72% compilation time)


14-element Vector{Any}:
 [0   0   0   0]
 [0   1   1   0]
 [0   0   1   0]
 [0   1   0   0]
 [0   1   1   1]
 [0   0   1   1]
 [0   0   0   1]
 [0   1   0   1]
 [0 1 0 0; 0 0 1 0]
 [0 1 0 0; 0 0 1 1]
 [0 0 1 0; 0 0 0 1]
 [0 1 0 1; 0 0 1 0]
 [0 1 1 0; 0 0 0 1]
 [0 1 0 1; 0 0 1 1]

## Project spaces in the standard way

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 x;0 1 0])
QM = q_matroid_from_matrix(Mat)

Q-Matroid of rank 2 in 3-dim. vector-space over the Galois field with characteristic 2

In [80]:
indeps = Q_Matroid_Independentspaces(QM)
deps = Q_Matroid_Dependentspaces(QM)
indeps

14-element Vector{Any}:
 [0   0   0]
 [1   1   0]
 [0   1   0]
 [1   0   0]
 [1   1   1]
 [0   1   1]
 [0   0   1]
 [1   0   1]
 [1 0 0; 0 1 0]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]

In [81]:
@time standard_projection(indeps)

  0.023083 seconds (4.98 k allocations: 317.333 KiB, 99.36% compilation time)


5-element Vector{Any}:
 [0   0]
 [1   1]
 [0   1]
 [1   0]
 [1 0; 0 1]

In [83]:
subs = subspaces_fix_space(indeps[9])
list = AbstractVector{Any}([])
for sub in subs
    for x in sub
        push!(list,x)
    end
end
standard_projection(list)

5-element Vector{Any}:
 [0   0]
 [1   1]
 [0   1]
 [1   0]
 [1 0; 0 1]

## Compute the vs-sum of two spaces

In [87]:
A = matrix(GF(3),[0 1 0 0])
B = matrix(GF(3),[1 0 0 0])
C = matrix(GF(3),[2 0 0 0])

[2   0   0   0]

In [88]:
@time sum_vs(A,B)

  0.000044 seconds (59 allocations: 4.070 KiB)


2-element Vector{gfp_mat}:
 [1   0   0   0]
 [0   1   0   0]

In [89]:
@time sum_vsV2(A,B)

  0.000020 seconds (23 allocations: 1.625 KiB)


[1   0   0   0]
[0   1   0   0]

In [90]:
@time multisum_vs([A,B,C])

  0.017221 seconds (11.89 k allocations: 831.269 KiB, 99.44% compilation time)


[1   0   0   0]
[0   1   0   0]

## Compute the vs-intersection of two spaces

In [91]:
A = matrix(GF(3),[1 0 0 0;0 1 0 0;0 0 1 0])
B = matrix(GF(3),[0 1 0 0;0 0 1 0;0 0 0 1])
@time inters_vs(A,B)

  0.194423 seconds (318.84 k allocations: 21.278 MiB, 99.46% compilation time)


[0   1   0   0]
[0   0   1   0]

In [92]:
@time inters_vsV2(A,B)

  0.084905 seconds (98.16 k allocations: 6.615 MiB, 99.62% compilation time)


[0   1   0   0]
[0   0   1   0]

In [93]:
@time inters_vsV3(A,B)

  0.008450 seconds (10.08 k allocations: 724.697 KiB, 99.41% compilation time)


[0   1   0   0]
[0   0   1   0]

## Compute the orthogonal complement of a space

In [94]:
A = matrix(GF(2),[1 0 0])

[1   0   0]

In [95]:
@time orthogonal_complement(A)

  0.050889 seconds (26.39 k allocations: 1.841 MiB, 99.66% compilation time)


1-element Vector{Any}:
 [0 1 0; 0 0 1]

In [96]:
@time orthogonal_complementV2(A)

  0.000021 seconds (33 allocations: 2.078 KiB)


1-element Vector{gfp_mat}:
 [0 1 0; 0 0 1]

## Compute all diamonds of a given list of spaces

In [97]:
all_subs = all_subspaces(GF(2),3)

16-element Vector{Any}:
 [0   0   0]
 [1   0   0]
 [0   1   0]
 [1   1   0]
 [0   0   1]
 [1   0   1]
 [0   1   1]
 [1   1   1]
 [1 0 0; 0 1 0]
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]
 [1 0 0; 0 1 0; 0 0 1]

In [98]:
deleteat!(all_subs,2)

15-element Vector{Any}:
 [0   0   0]
 [0   1   0]
 [1   1   0]
 [0   0   1]
 [1   0   1]
 [0   1   1]
 [1   1   1]
 [1 0 0; 0 1 0]
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]
 [1 0 0; 0 1 0; 0 0 1]

In [99]:
@time diamond_list(all_subs)

  0.740721 seconds (1.41 M allocations: 88.726 MiB, 4.73% gc time, 99.68% compilation time)


10-element Vector{AbstractVector{Any}}:
 [[0 0 0], [0 1 0], [0 0 1], [0 1 1], [0 1 0; 0 0 1]]
 [[0 0 0], [0 1 0], [1 0 1], [1 1 1], [1 0 1; 0 1 0]]
 [[0 0 0], [1 1 0], [0 0 1], [1 1 1], [1 1 0; 0 0 1]]
 [[0 0 0], [1 1 0], [1 0 1], [0 1 1], [1 0 1; 0 1 1]]
 [[0 1 0], [1 0 0; 0 1 0], [1 0 1; 0 1 0], [0 1 0; 0 0 1], [1 0 0; 0 1 0; 0 0 1]]
 [[1 1 0], [1 0 0; 0 1 0], [1 0 1; 0 1 1], [1 1 0; 0 0 1], [1 0 0; 0 1 0; 0 0 1]]
 [[0 0 1], [1 1 0; 0 0 1], [0 1 0; 0 0 1], [1 0 0; 0 0 1], [1 0 0; 0 1 0; 0 0 1]]
 [[1 0 1], [1 0 1; 0 1 1], [1 0 1; 0 1 0], [1 0 0; 0 0 1], [1 0 0; 0 1 0; 0 0 1]]
 [[0 1 1], [1 0 1; 0 1 1], [0 1 0; 0 0 1], [1 0 0; 0 1 1], [1 0 0; 0 1 0; 0 0 1]]
 [[1 1 1], [1 1 0; 0 0 1], [1 0 1; 0 1 0], [1 0 0; 0 1 1], [1 0 0; 0 1 0; 0 0 1]]

## Compute the all lattice k_intervals of a list of spaces

In [100]:
all_subs = all_subspaces(GF(2),4)

67-element Vector{Any}:
 [0   0   0   0]
 [1   0   0   0]
 [0   1   0   0]
 [1   1   0   0]
 [0   0   1   0]
 [1   0   1   0]
 [0   1   1   0]
 [1   1   1   0]
 [0   0   0   1]
 [1   0   0   1]
 ⋮
 [0 1 0 0; 0 0 1 0; 0 0 0 1]
 [1 0 0 1; 0 1 0 0; 0 0 1 0]
 [1 0 1 0; 0 1 0 0; 0 0 0 1]
 [1 0 0 1; 0 1 0 0; 0 0 1 1]
 [1 1 0 0; 0 0 1 0; 0 0 0 1]
 [1 0 0 1; 0 1 0 1; 0 0 1 0]
 [1 0 1 0; 0 1 1 0; 0 0 0 1]
 [1 0 0 1; 0 1 0 1; 0 0 1 1]
 [1 0 0 0; 0 1 0 0; 0 0 1 0; 0 0 0 1]

In [ ]:
all_subs[17]

In [166]:
deleteat!(all_subs,2:10)

58-element Vector{Any}:
 [0   0   0   0]
 [0   1   0   1]
 [1   1   0   1]
 [0   0   1   1]
 [1   0   1   1]
 [0   1   1   1]
 [1   1   1   1]
 [1 0 0 0; 0 1 0 0]
 [1 0 0 0; 0 0 1 0]
 [1 0 0 0; 0 1 1 0]
 ⋮
 [0 1 0 0; 0 0 1 0; 0 0 0 1]
 [1 0 0 1; 0 1 0 0; 0 0 1 0]
 [1 0 1 0; 0 1 0 0; 0 0 0 1]
 [1 0 0 1; 0 1 0 0; 0 0 1 1]
 [1 1 0 0; 0 0 1 0; 0 0 0 1]
 [1 0 0 1; 0 1 0 1; 0 0 1 0]
 [1 0 1 0; 0 1 1 0; 0 0 0 1]
 [1 0 0 1; 0 1 0 1; 0 0 1 1]
 [1 0 0 0; 0 1 0 0; 0 0 1 0; 0 0 0 1]

In [101]:
k_ints = k_interval(all_subs,3)

30-element Vector{AbstractVector{Any}}:
 [[0 0 0 0], [1 0 0 0], [0 1 0 0], [1 1 0 0], [0 0 1 0], [1 0 1 0], [0 1 1 0], [1 1 1 0], [1 0 0 0; 0 1 0 0], [1 0 0 0; 0 0 1 0], [1 0 0 0; 0 1 1 0], [0 1 0 0; 0 0 1 0], [1 0 1 0; 0 1 0 0], [1 1 0 0; 0 0 1 0], [1 0 1 0; 0 1 1 0], [1 0 0 0; 0 1 0 0; 0 0 1 0]]
 [[0 0 0 0], [1 0 0 0], [0 1 0 0], [1 1 0 0], [0 0 0 1], [1 0 0 1], [0 1 0 1], [1 1 0 1], [1 0 0 0; 0 1 0 0], [1 0 0 0; 0 0 0 1], [1 0 0 0; 0 1 0 1], [0 1 0 0; 0 0 0 1], [1 0 0 1; 0 1 0 0], [1 1 0 0; 0 0 0 1], [1 0 0 1; 0 1 0 1], [1 0 0 0; 0 1 0 0; 0 0 0 1]]
 [[0 0 0 0], [1 0 0 0], [0 1 0 0], [1 1 0 0], [0 0 1 1], [1 0 1 1], [0 1 1 1], [1 1 1 1], [1 0 0 0; 0 1 0 0], [1 0 0 0; 0 0 1 1], [1 0 0 0; 0 1 1 1], [0 1 0 0; 0 0 1 1], [1 0 1 1; 0 1 0 0], [1 1 0 0; 0 0 1 1], [1 0 1 1; 0 1 1 1], [1 0 0 0; 0 1 0 0; 0 0 1 1]]
 [[0 0 0 0], [1 0 0 0], [0 0 1 0], [1 0 1 0], [0 0 0 1], [1 0 0 1], [0 0 1 1], [1 0 1 1], [1 0 0 0; 0 0 1 0], [1 0 0 0; 0 0 0 1], [1 0 0 0; 0 0 1 1], [0 0 1 0; 0 0 0 1], [1 0 0 1; 0 0

In [102]:
length(k_ints[1])

16

##
***

## Binary encoding of a list of spaces

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 x 0])
QM = q_matroid_from_matrix(Mat)
Bases = QM.bases
all_sp = subspaces_fix_dim(GF(Int(characteristic(Ext_F))),rank(Bases[1]),ncols(Bases[1]))

15-element Vector{Any}:
 [1   0   0   0]
 [0   1   0   0]
 [1   1   0   0]
 [0   0   1   0]
 [1   0   1   0]
 [0   1   1   0]
 [1   1   1   0]
 [0   0   0   1]
 [1   0   0   1]
 [0   1   0   1]
 [1   1   0   1]
 [0   0   1   1]
 [1   0   1   1]
 [0   1   1   1]
 [1   1   1   1]

In [104]:
Bases

12-element Vector{Any}:
 [1   0   0   0]
 [1   1   0   0]
 [0   0   1   0]
 [1   0   1   0]
 [0   1   1   0]
 [1   1   1   0]
 [1   0   0   1]
 [1   1   0   1]
 [0   0   1   1]
 [1   0   1   1]
 [0   1   1   1]
 [1   1   1   1]

In [105]:
s = join(binary_encoding(Bases,all_sp))

"101111101011111"

## Encoded and decoded a list of spaces/encodings

In [ ]:
spaces = subspaces_fix_dim(GF(2),2,2)
zero_vec = matrix(GF(2),[0 0])
Ext_F,x = finite_field(2,3,"x")
A = matrix(Ext_F,[1 0])
QM = q_matroid_from_matrix(A)
indeps = Q_Matroid_Independentspaces(QM)

3-element Vector{Any}:
 [0   0]
 [1   0]
 [1   1]

In [108]:
all_ones = subspaces_fix_dim(GF(2),1,2)

3-element Vector{Any}:
 [1   0]
 [0   1]
 [1   1]

In [109]:
ones_dict = OrderedDict([id=>elm for (id,elm) in enumerate(all_ones)])

OrderedDict{Int64, gfp_mat} with 3 entries:
  1 => [1 0]
  2 => [0 1]
  3 => [1 1]

In [110]:
@time ens = sub_encoding(indeps,ones_dict,true)

  0.100330 seconds (214.67 k allocations: 14.341 MiB, 99.91% compilation time)


(AbstractVector{Int64}[[0], [0, 1], [0, 3]], 2)

In [111]:
@time des = sub_decoding(ens[1],ones_dict)

  0.242905 seconds (519.81 k allocations: 31.493 MiB, 99.40% compilation time)


3-element Vector{Any}:
 [0   0]
 [1   0]
 [1   1]

In [112]:
des == indeps

true

## Compute encoded sub-and superspaces of a given subspace

In [ ]:
all = all_subspaces(GF(2),3)
ones_dict = OrderedDict([id-1=>elm for (id,elm) in enumerate(all) if rank(elm)==1])
encoded_all = sub_encoding(all,ones_dict)

In [ ]:
encoded_space1 = [0, 1]
@time en_supers = encoded_containments_fix_space(encoded_space1,encoded_all)

In [ ]:
encoded_space2 = [0, 1, 2, 3]
@time en_subs = encoded_subspaces_fix_space(encoded_space2,encoded_all)

In [ ]:
sub_decoding(en_supers,ones_dict)

In [ ]:
sub_decoding(en_subs,ones_dict)

## Compute the all lattice k_intervals of a list of encoded-spaces

In [ ]:
all = all_subspaces(GF(2),4)
ones_dict = OrderedDict([id-1=>elm for (id,elm) in enumerate(all) if rank(elm)==1])
encoded_all, q = sub_encoding(all,ones_dict,true)
encoded_spaces = copy(encoded_all)

In [17]:
deleteat!(encoded_spaces,2:10);

In [ ]:
@time con_3_int = encoded_k_interval(encoded_spaces,q,3,encoded_all)

# Computing Independent/Dependent spaces
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
A = matrix(Ext_F,[1 0 0;0 1 0])
QM1 = q_matroid_from_matrix(A)
bases = QM1.bases

4-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 1 1]
 [1 0 1; 0 1 0]
 [1 0 1; 0 1 1]

In [114]:
id_mat = matrix(GF(2),[1 0 0;0 1 0;0 0 1])
QM = Q_Matroid(id_mat,bases)

Q-Matroid of rank 2 in 3-dim. vector-space over the Galois field with characteristic 2

In [116]:
Indeps = Q_Matroid_Independentspaces(QM)

11-element Vector{Any}:
 [0   0   0]
 [1   1   0]
 [0   1   0]
 [1   0   0]
 [1   1   1]
 [0   1   1]
 [1   0   1]
 [1 0 0; 0 1 0]
 [1 0 0; 0 1 1]
 [1 0 1; 0 1 0]
 [1 0 1; 0 1 1]

In [117]:
Q_Matroid_Dependentspaces(QM)

5-element Vector{Any}:
 [0   0   1]
 [1 0 0; 0 0 1]
 [0 1 0; 0 0 1]
 [1 1 0; 0 0 1]
 [1 0 0; 0 1 0; 0 0 1]

In [118]:
Q_Matroid_Loopspace(QM)

1-element Vector{Any}:
 [0   0   1]

# Creating Q-Matroid from Matrix
***
***

In [ ]:
Ext_F,x = finite_field(2,4,"x")
Mat = matrix(Ext_F,[1 0 0 x;0 1 x^2 x])
QM = q_matroid_from_matrix(Mat)

Q-Matroid of rank 2 in 4-dim. vector-space over the Galois field with characteristic 2

In [4]:
QM.bases

33-element Vector{Any}:
 [1 0 0 0; 0 1 0 0]
 [1 0 0 0; 0 0 1 0]
 [1 0 0 0; 0 1 1 0]
 [1 0 0 0; 0 0 0 1]
 [1 0 0 0; 0 1 0 1]
 [1 0 0 0; 0 0 1 1]
 [1 0 0 0; 0 1 1 1]
 [1 0 1 0; 0 1 0 0]
 [0 1 0 0; 0 0 0 1]
 [1 0 0 1; 0 1 0 0]
 ⋮
 [1 0 1 0; 0 1 1 1]
 [0 1 1 0; 0 0 0 1]
 [1 0 0 1; 0 1 1 0]
 [0 1 0 1; 0 0 1 1]
 [1 0 1 1; 0 1 1 0]
 [1 1 1 0; 0 0 0 1]
 [1 0 0 1; 0 1 1 1]
 [1 0 1 1; 0 1 0 1]
 [1 1 0 1; 0 0 1 1]

In [121]:
Q_Matroid_Independentspaces(QM)

5-element Vector{Any}:
 [0   0   0]
 [1   0   0]
 [1   1   0]
 [1   0   1]
 [1   1   1]

In [122]:
Q_Matroid_Dependentspaces(QM)

11-element Vector{Any}:
 [0   1   0]
 [0   0   1]
 [0   1   1]
 [1 0 0; 0 1 0]
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]
 [1 0 0; 0 1 0; 0 0 1]

In [123]:
Q_Matroid_Loopspace(QM)

3-element Vector{Any}:
 [0   1   0]
 [0   0   1]
 [0   1   1]

# Rank Value
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 0;0 1 0])
QM = q_matroid_from_matrix(Mat)
indeps = Q_Matroid_Independentspaces(QM)
deps = Q_Matroid_Dependentspaces(QM)

5-element Vector{Any}:
 [0   0   1]
 [1 0 0; 0 0 1]
 [0 1 0; 0 0 1]
 [1 1 0; 0 0 1]
 [1 0 0; 0 1 0; 0 0 1]

In [125]:
indeps

11-element Vector{Any}:
 [0   0   0]
 [1   1   0]
 [0   1   0]
 [1   0   0]
 [1   1   1]
 [0   1   1]
 [1   0   1]
 [1 0 0; 0 1 0]
 [1 0 0; 0 1 1]
 [1 0 1; 0 1 0]
 [1 0 1; 0 1 1]

In [126]:
A = matrix(GF(2),[1 0 0;0 0 1])
Q_Matroid_Ranks(QM,A)

1

In [127]:
Q_Matroid_Ranks(QM,QM.groundspace,indeps,deps)

2

# Computing Circuits
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 0; 0 1 0])
QM = q_matroid_from_matrix(Mat)
length(QM.bases)

4

In [131]:
QM.bases

4-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 1 1]
 [1 0 1; 0 1 0]
 [1 0 1; 0 1 1]

In [132]:
@time Q_Matroid_Circuits(QM)

  0.000375 seconds (5.24 k allocations: 266.711 KiB)


1-element Vector{Any}:
 [0   0   1]

In [133]:
@time Q_Matroid_CircuitsV2(QM)

  0.000309 seconds (5.22 k allocations: 266.117 KiB)


1-element Vector{Any}:
 [0   0   1]

# Checking for paving q-matroids
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 0;0 1 0;0 0 1])
QM = q_matroid_from_matrix(Mat)

Q-Matroid of rank 3 in 3-dim. vector-space over the Galois field with characteristic 2

In [135]:
QM.bases

1-element Vector{Any}:
 [1 0 0; 0 1 0; 0 0 1]

In [136]:
Q_Matroid_Dependentspaces(QM)

Any[]

In [137]:
Q_Matroid_Circuits(QM)

Any[]

In [138]:
Is_paving_q_matroid(QM)

true

# Computing the q-matroid lattice
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 0])
QM = q_matroid_from_matrix(Mat)
indeps = Q_Matroid_Independentspaces(QM)
deps = Q_Matroid_Dependentspaces(QM);

In [ ]:
Q_Matroid_lattice(QM,indeps,deps,"yes")

In [141]:
G = Q_Matroid_latticeV2(QM,indeps,deps)
collect(Oscar.edges(G))

11-element Vector{Oscar.Edge}:
 Oscar.Edge(2, 1)
 Oscar.Edge(4, 1)
 Oscar.Edge(6, 1)
 Oscar.Edge(8, 1)
 Oscar.Edge(9, 3)
 Oscar.Edge(10, 5)
 Oscar.Edge(11, 7)
 Oscar.Edge(13, 3)
 Oscar.Edge(14, 5)
 Oscar.Edge(15, 7)
 Oscar.Edge(16, 12)

# Check if two q-matroids are isomorphic
***
***

In [ ]:
Ext_F1,x = finite_field(2,3,"x")
Ext_F2,y = finite_field(2,3,"y")
Mat1 = matrix(Ext_F1,[1 0 0 0 0;0 1 x 0 0])
Mat2 = matrix(Ext_F1,[1 0 0 0 0;0 1 x 0 0])
QM1 = q_matroid_from_matrix(Mat1)
QM2 = q_matroid_from_matrix(Mat2)
Indeps1 = Q_Matroid_Independentspaces(QM1) 
Indeps2 = Q_Matroid_Independentspaces(QM2)
Deps1 = Q_Matroid_Dependentspaces(QM1)
Deps2 = Q_Matroid_Dependentspaces(QM2)
l1 = Q_Matroid_lattice(QM1,Indeps1,Deps1,"no")
l2 = Q_Matroid_lattice(QM2,Indeps2,Deps2,"no")

{374, 688} undirected simple Int64 graph

In [330]:
Are_isom_q_matroids(QM1,QM2)

true

In [331]:
Are_isom_q_matroidsV2(QM1,QM2)

true

In [332]:
Are_isom_q_matroidsV2(QM1,QM2,[l1,l2])

true

# Compute isom-classes from list of matrices
***
***

In [ ]:
Ext_F,x = finite_field(2,1,"x")
mats = matrix_collec(Ext_F,2,4)

16-element Vector{Any}:
 [1 0 0 0; 0 1 1 0]
 [1 0 0 1; 0 1 1 0]
 [1 0 0 1; 0 1 1 1]
 [1 0 0 0; 0 1 1 1]
 [1 0 0 1; 0 1 0 0]
 [1 0 0 1; 0 1 0 1]
 [1 0 0 0; 0 1 0 1]
 [1 0 0 0; 0 1 0 0]
 [1 0 1 1; 0 1 0 1]
 [1 0 1 0; 0 1 0 1]
 [1 0 1 0; 0 1 0 0]
 [1 0 1 1; 0 1 0 0]
 [1 0 1 0; 0 1 1 1]
 [1 0 1 0; 0 1 1 0]
 [1 0 1 1; 0 1 1 0]
 [1 0 1 1; 0 1 1 1]

In [ ]:
isom1 = Isom_classes_from_mats(mats)

In [19]:
isom2 = Isom_classes_from_matsV2(mats)

1-element Vector{Any}:
 ([1 0 0 0; 0 1 1 0], Q-Matroid of rank 2 in 4-dim. vector-space over the Galois field with characteristic 2, 16, 15)

# Compute isom-classes from list of bases
***
***

In [ ]:
Ext_F1,x1 = finite_field(2,3,"x1")
Ext_F2,x2 = finite_field(2,3,"x2")
Ext_F3,x3 = finite_field(2,3,"x3")
Ext_F4,x4 = finite_field(2,3,"x4")
mat1 = matrix(Ext_F1,[1 0 1])
mat2 = matrix(Ext_F2,[1 0 1])
mat3 = matrix(Ext_F3,[1 1 0])
mat4 = matrix(Ext_F4,[1 x4 x4^2])
QM1 = q_matroid_from_matrix(mat1)
QM2 = q_matroid_from_matrix(mat2)
QM3 = q_matroid_from_matrix(mat3)
QM4 = q_matroid_from_matrix(mat4)
list = AbstractVector{AbstractVector{Any}}([QM1.bases,QM2.bases,QM3.bases,QM4.bases])

4-element Vector{AbstractVector{Any}}:
 [[1 0 0], [1 1 0], [0 0 1], [0 1 1]]
 [[1 0 0], [1 1 0], [0 0 1], [0 1 1]]
 [[1 0 0], [0 1 0], [1 0 1], [0 1 1]]
 [[1 0 0], [0 1 0], [1 1 0], [0 0 1], [1 0 1], [0 1 1], [1 1 1]]

In [152]:
Isom_classes_from_bases(list)

2-element Vector{Any}:
 (Q-Matroid of rank 1 in 3-dim. vector-space over the Galois field with characteristic 2, 4, 1)
 (Q-Matroid of rank 1 in 3-dim. vector-space over the Galois field with characteristic 2, 7, 0)

In [153]:
Isom_classes_from_basesV2(list)

2-element Vector{Any}:
 (Q-Matroid of rank 1 in 3-dim. vector-space over the Galois field with characteristic 2, 4, 1)
 (Q-Matroid of rank 1 in 3-dim. vector-space over the Galois field with characteristic 2, 7, 0)

# Compute the characteristic polynomial of a q-matroid
***
***

In [ ]:
Ext_F1,x = finite_field(2,3,"x")
Ext_F2,y = finite_field(2,1,"y")
Ext_F3,w = finite_field(2,2,"w")
Mat1 = matrix(Ext_F1,[1 0 0;0 1 x])
Mat2 = matrix(Ext_F2,[[1]])
Mat3 = matrix(Ext_F3,[1 w])
QM1 = q_matroid_from_matrix(Mat1)
QM2 = q_matroid_from_matrix(Mat2)
QM3 = q_matroid_from_matrix(Mat3)
indeps1 = Q_Matroid_Independentspaces(QM1) 
deps1 = Q_Matroid_Dependentspaces(QM1)
indeps2 = Q_Matroid_Independentspaces(QM2) 
deps2 = Q_Matroid_Dependentspaces(QM2)
indeps3 = Q_Matroid_Independentspaces(QM3) 
deps3 = Q_Matroid_Dependentspaces(QM3)

1-element Vector{Any}:
 [1 0; 0 1]

In [383]:
polyn1 = Q_Matroid_charpoly(QM1,indeps1,deps1)
polyn2 = Q_Matroid_charpoly(QM2,indeps2,deps2)
polyn3 = Q_Matroid_charpoly(QM3,indeps3,deps3)
polyn1,polyn2,polyn3

(z^2 - 5*z + 4, z - 1, z - 1)

# Compute the closure of a space in a given q-matroid
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 0;0 1 0])
QM = q_matroid_from_matrix(Mat)
A = matrix(GF(2),[1 0 0;0 1 0])

[1   0   0]
[0   1   0]

In [170]:
Q_Matroid_Closure_Function(QM,A)

[1   0   0]
[0   1   0]
[0   0   1]

# Compute the cyclic closure of a space in a given q-matroid
***
***

In [ ]:
Ext_F,x = finite_field(2,1,"x")
Mat = matrix(Ext_F,[1 1])
QM = q_matroid_from_matrix(Mat)
A = matrix(GF(2),[1 1])

[1   1]

In [172]:
Q_Matroid_CyclicClosure_Function(QM,QM.groundspace)

[1   1]

# Check if a given q-matroid is full
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 x x^2])
QM = q_matroid_from_matrix(Mat)

Q-Matroid of rank 1 in 3-dim. vector-space over the Galois field with characteristic 2

In [174]:
Is_full_q_matroid(QM)

true

# Compute the flats of a q-matroid
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 0;0 1 0])
QM = q_matroid_from_matrix(Mat)

Q-Matroid of rank 2 in 3-dim. vector-space over the Galois field with characteristic 2

In [178]:
Q_Matroid_Flats(QM)

5-element Vector{Any}:
 [0   0   1]
 [1 0 0; 0 0 1]
 [0 1 0; 0 0 1]
 [1 1 0; 0 0 1]
 [1 0 0; 0 1 0; 0 0 1]

# Compute the hyperplanes of a q-matroid
***
***

In [ ]:
Ext_F,x = finite_field(2,4,"x")
Mat = matrix(Ext_F,[1 0 0 0; 0 1 x 0])
QM = q_matroid_from_matrix(Mat)
length(QM.bases),Is_paving_q_matroid(QM)

(24, false)

In [180]:
Q_Matroid_Flats(QM)

7-element Vector{Any}:
 [0   0   0   1]
 [1 0 0 0; 0 0 0 1]
 [1 1 0 0; 0 0 0 1]
 [1 0 1 0; 0 0 0 1]
 [1 1 1 0; 0 0 0 1]
 [0 1 0 0; 0 0 1 0; 0 0 0 1]
 [1 0 0 0; 0 1 0 0; 0 0 1 0; 0 0 0 1]

In [181]:
Q_Matroid_Hyperplanes(QM)

5-element Vector{Any}:
 [1 0 0 0; 0 0 0 1]
 [1 1 0 0; 0 0 0 1]
 [1 0 1 0; 0 0 0 1]
 [1 1 1 0; 0 0 0 1]
 [0 1 0 0; 0 0 1 0; 0 0 0 1]

# Compute the circuit-hyperplanes of a q-matroid
***
***

In [ ]:
Ext_F,x = finite_field(2,4,"x")
Mat = matrix(Ext_F,[1 0 0 x^2+x; 0 1 x^2+x x^2+x+1])
QM = q_matroid_from_matrix(Mat)
length(QM.bases),Is_paving_q_matroid(QM)

(30, true)

In [42]:
CH = Q_Matroid_CircuitHyperplanes(QM)

5-element Vector{Any}:
 [1 0 0 0; 0 1 1 1]
 [0 1 0 0; 0 0 1 0]
 [1 0 0 1; 0 1 0 1]
 [1 0 1 0; 0 0 0 1]
 [1 1 0 1; 0 0 1 1]

In [43]:
for combi in combinations(CH,2)
    println(inters_vsV3(combi[1],combi[2]))
end

[0 0 0 0]
[0 0 0 0]
[0 0 0 0]
[0 0 0 0]
[0 0 0 0]
[0 0 0 0]
[0 0 0 0]
[0 0 0 0]
[0 0 0 0]
[0 0 0 0]


# Compute the cyclic flats of a q-matroid
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 x x^2])
QM = q_matroid_from_matrix(Mat)

Q-Matroid of rank 1 in 3-dim. vector-space over the Galois field with characteristic 2

In [183]:
Q_Matroid_Flats(QM)

2-element Vector{Any}:
 [0   0   0]
 [1 0 0; 0 1 0; 0 0 1]

In [184]:
Q_Matroid_Openspaces(QM)

9-element Vector{Any}:
 [1 0 0; 0 1 0; 0 0 1]
 [0 1 0; 0 0 1]
 [1 0 0; 0 0 1]
 [1 1 0; 0 0 1]
 [1 0 0; 0 1 0]
 [1 0 1; 0 1 0]
 [1 0 0; 0 1 1]
 [1 0 1; 0 1 1]
 [0   0   0]

In [185]:
Q_Matroid_CircuitsV2(QM)

7-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]

In [186]:
Q_Matroid_CyclicFlats(QM)

2-element Vector{Any}:
 [0   0   0]
 [1 0 0; 0 1 0; 0 0 1]

# Check if a list of spaces are the Dependent-spaces of a q-Matroid
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 0; 0 1 0])
QM = q_matroid_from_matrix(Mat)
Deps = Q_Matroid_Dependentspaces(QM)

5-element Vector{Any}:
 [0   0   1]
 [1 0 0; 0 0 1]
 [0 1 0; 0 0 1]
 [1 1 0; 0 0 1]
 [1 0 0; 0 1 0; 0 0 1]

In [337]:
#deleteat!(Deps,length(Deps))
deleteat!(Deps,1)

4-element Vector{Any}:
 [1 0 0; 0 0 1]
 [0 1 0; 0 0 1]
 [1 1 0; 0 0 1]
 [1 0 0; 0 1 0; 0 0 1]

In [338]:
Are_q_matroid_dependentspaces(Deps)

false

# Check if a list of spaces are the Hyperlanes of a q-Matroid
***
***

In [ ]:
Ext_F,x = finite_field(2,4,"x")
Mat = matrix(Ext_F,[1 0 0 0; 0 1 1 0])
QM = q_matroid_from_matrix(Mat)

Q-Matroid of rank 2 in 4-dim. vector-space over the Galois field with characteristic 2

In [341]:
Hyperps = Q_Matroid_Hyperplanes(QM)

3-element Vector{Any}:
 [1 0 0 0; 0 1 1 0; 0 0 0 1]
 [0 1 0 0; 0 0 1 0; 0 0 0 1]
 [1 0 1 0; 0 1 1 0; 0 0 0 1]

In [342]:
deleteat!(Hyperps,[1])
#insert!(Hyperps,1,matrix(GF(2),[1 0 0;0 1 0;0 0 1]))

2-element Vector{Any}:
 [0 1 0 0; 0 0 1 0; 0 0 0 1]
 [1 0 1 0; 0 1 1 0; 0 0 0 1]

In [343]:
Are_q_matroid_hyperplanes(Hyperps, "Yes")

(false, Any["Axiom (H3)", gfp_mat[[0 1 0 0; 0 0 1 0; 0 0 0 1], [1 0 1 0; 0 1 1 0; 0 0 0 1]]])

# Check if a list of spaces are the bases of a q-Matroid
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 x;0 1 x^2])
QM = q_matroid_from_matrix(Mat)
Bases = QM.bases

7-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]

In [9]:
[Bases[1]]

1-element Vector{gfp_mat}:
 [1 0 0; 0 1 0]

In [6]:
deleteat!(Bases,1)

3-element Vector{Any}:
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]

In [15]:
Are_q_matroid_bases([Bases[7]])

false

# Computing the projectivization matroid of a q-matroid
***
***

In [ ]:
q_binomcoeff(3,4,1)

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 0;0 1 0])
QM = q_matroid_from_matrix(Mat)
QM.bases

4-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 1 1]
 [1 0 1; 0 1 0]
 [1 0 1; 0 1 1]

In [205]:
proj_mat = Projectivization_matroid(QM)

Matroid of rank 2 on 7 elements

In [ ]:
B = [[1,2],[1,3],[1,4],[2,3],[2,4]]

M = matroid_from_bases(B,4)

# Compute the simplified matrix
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 x;0 1 x^2])
QM = q_matroid_from_matrix(Mat)
QM.bases

7-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]

In [213]:
deleteat!(QM.bases,1)

6-element Vector{Any}:
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]

In [214]:
print(Simplyfy_rep_mat(QM)[1])

[1 x1 0; 0 0 1]

# Check if a given q-matroid is representable
***
***

In [ ]:
Ext_F,x = finite_field(2,4,"x")
Mat = matrix(Ext_F,[1 0])
QM = q_matroid_from_matrix(Mat)
QM.bases

2-element Vector{Any}:
 [1   0]
 [1   1]

In [216]:
rep, I = Is_representable(QM)

("Q-Matroid is representable!!", ideal(x3 + 1, x1))

In [217]:
groebner_basis(I)

Gröbner basis with elements
1 -> x3 + 1
2 -> x1
with respect to the ordering
degrevlex([x1, x2, x3])

In [218]:
Simplyfy_rep_mat(QM)

([1 x1], Multivariate Polynomial Ring in x1, x2, x3 over Finite field of degree 1 over F_2, x3)

***

In [16]:
two_spaces = subspaces_fix_dim(GF(2),2,4);

In [21]:
new_list = AbstractVector{fpMatrix}([])
A = matrix(GF(2),[1 0 0 0;0 1 0 0])
B = matrix(GF(2),[0 0 1 0;0 0 0 1])
C = matrix(GF(2),[1 0 0 1;0 1 1 0])
D = matrix(GF(2),[1 0 1 1;0 1 0 1])
l = [A,B,C,D]
for space in two_spaces
    if !(space in l)
        push!(new_list,space)
    end
end
new_list;

In [ ]:
qm = Q_Matroid(matrix(GF(2),[1 0 0 0;0 1 0 0;0 0 1 0;0 0 0 1]),new_list)

In [ ]:
Is_representable(qm)

# Compute the dual Q-Matroid
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 x 0])
QM = q_matroid_from_matrix(Mat)
QM.bases

6-element Vector{Any}:
 [1   0   0]
 [0   1   0]
 [1   1   0]
 [1   0   1]
 [0   1   1]
 [1   1   1]

In [220]:
DQM = Dual_Q_Matroid(QM)
DQM.bases

6-element Vector{Any}:
 [0 1 0; 0 0 1]
 [1 0 0; 0 0 1]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 0 0; 0 1 1]
 [1 0 1; 0 1 1]

# Compute the q-matroid base polytope
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat =  matrix(Ext_F,[1 0 0])
QM = q_matroid_from_matrix(Mat)
Bases = QM.bases;
length(Bases), Bases

(4, Any[[1 0 0], [1 1 0], [1 0 1], [1 1 1]])

In [355]:
QM = Uniform_q_matroid(GF(2),2,3)
Bases = QM.bases;
length(Bases)

7

In [ ]:
#= bases31 = AbstractVector{fpMatrix}([])
two_spaces = subspaces_fix_dim(GF(2),2,4)
A = matrix(GF(2),[1 0 0 0;0 1 0 0])
B = matrix(GF(2),[0 0 1 0;0 0 0 1])
C = matrix(GF(2),[1 0 0 1;0 1 1 0])
D = matrix(GF(2),[1 0 1 0;0 1 1 1])
l = [A,B,C,D]
for space in two_spaces
    if !(space in l)
        push!(bases31,space)
    end
end
qm31 = Q_Matroid(matrix(GF(2),[1 0 0 0;0 1 0 0;0 0 1 0;0 0 0 1]),bases31) =#

In [357]:
P = Q_Matroid_Base_polytope(QM,true)

(A polyhedron in ambient dimension 7, Vector{Any}[[1, [1 0 0]], [2, [0 1 0]], [3, [1 1 0]], [4, [0 0 1]], [5, [1 0 1]], [6, [0 1 1]], [7, [1 1 1]]])

In [358]:
dim(P[1])

3

In [359]:
affine_hull(P[1])

4-element SubObjectIterator{AffineHyperplane{fmpq}} over the Hyperplanes of R^7 described by:
x₂ = 0
x₄ = 0
x₆ = 0
x₁ + x₃ + x₅ + x₇ = 1


In [360]:
print_constraints(P[1])

-x₁ ≦ 0
-x₃ ≦ 0
x₁ + x₃ + x₇ ≦ 1
-x₇ ≦ 0


In [361]:
F = Q_Matroid_Flats(QM)

2-element Vector{Any}:
 [0 1 0; 0 0 1]
 [1 0 0; 0 1 0; 0 0 1]

# Compute the q-matroid polytope
***
***

In [20]:
function Q_Matroid_Polytope(QM)
    Field = base_ring(QM.groundspace)
    q = Int(characteristic(Field))
    dim = ncols(QM.groundspace)
    r = rank(QM.bases[1])
    q_r = q_binomcoeff(q,r,1)
    ambient_dim = q_binomcoeff(q,dim,1)

    # Inequalities for the r-simplex
    id = zeros(Int,ambient_dim,ambient_dim)

    for i in range(1,ambient_dim)
        for j in range(1,ambient_dim)
            if i == j
                id[i,j]=1
            end
        end
    end
    neg_id = -id
    one_vec = Array(ones(Int,1,ambient_dim))

    # Define A, b for the polytope
    A = Array(vcat(neg_id,one_vec,-one_vec))
    b = Array(vcat(zeros(Int,ambient_dim),q_r,-q_r))

    # Label all 1-spaces
    one_spaces = subspaces_fix_dim(Field,1,dim)                                 # 1-spaces are computed twice
    labeled_one_spaces = [[id,elm] for (id,elm) in enumerate(one_spaces)]

    # Inequalities coming from the ranks of the subspaces
    all_subs = all_subspaces(Field,dim)                                         # 1-spaces are computed twice
    ones_r_spaces = []

    for X in all_subs
        zero_vec = zeros(Int,1,ambient_dim)
        rank_X = Q_Matroid_Ranks(QM,X)
        one_X = dim_one_subs(X)
        for y in one_X
            for elm in labeled_one_spaces
                if y == elm[2]
                    zero_vec[elm[1]] = 1
                end
            end
        end

        if rank(X)== r
            push!(ones_r_spaces,Set(one_X))
        end

        A = Array(vcat(A,Array(zero_vec)))
        q_X = q_binomcoeff(q,rank_X,1)
        b = Array(vcat(b,Array([q_X])))
    end

    #= # Collect all qr-tuples of 1-spaces which do not form a r-space
    qr_tuples = collect(combinations(one_spaces,q_r))
    S = [elm for elm in qr_tuples if !(Set(elm) in ones_r_spaces)]

    # Equalities coming from these q_r-tuples of 1-spaces
    for tup in S
        zero_vec = zeros(Int,1,ambient_dim)
        for y in tup
            for elm in labeled_one_spaces
                if y == elm[2]
                    zero_vec[elm[1]] = 1
                end
            end
        end
        A = Array(vcat(A,Array(zero_vec),Array(-zero_vec)))
        b = Array(vcat(b,Array([0]),Array([0])))
    end =#

    return polyhedron((A,b))
    
end

Q_Matroid_Polytope (generic function with 1 method)

In [ ]:
Ext_F,x = finite_field(2,3,"x")
A = matrix(Ext_F,[1 0 0;0 1 x])
QM = q_matroid_from_matrix(A)
UM = Uniform_q_matroid(GF(2),3,4)
P_bar = Q_Matroid_Polytope(QM)
P = Q_Matroid_Base_polytope(QM);

In [363]:
dim(P_bar),dim(P),length(Oscar.vertices(P_bar)),length(Oscar.vertices(P))

(6, 5, 22, 6)

In [364]:
[v  for v in Oscar.vertices(P_bar) if !(v in Oscar.vertices(P) )]

16-element Vector{PointVector{fmpq}}:
 [0, 0, 0, 1, 1, 0, 1]
 [0, 0, 0, 0, 1, 1, 1]
 [0, 0, 1, 0, 1, 0, 1]
 [0, 0, 1, 0, 0, 1, 1]
 [0, 0, 1, 1, 1, 0, 0]
 [0, 1, 1, 0, 1, 0, 0]
 [0, 1, 1, 0, 0, 0, 1]
 [1, 0, 1, 0, 0, 0, 1]
 [1, 0, 0, 1, 0, 0, 1]
 [1, 0, 0, 0, 1, 1, 0]
 [1, 0, 1, 0, 1, 0, 0]
 [1, 0, 0, 0, 1, 0, 1]
 [1, 0, 1, 0, 0, 1, 0]
 [1, 1, 0, 0, 1, 0, 0]
 [1, 0, 1, 1, 0, 0, 0]
 [1, 1, 0, 0, 0, 0, 1]

In [15]:
Set(collect(Oscar.vertices(P_bar)))==Set(collect(Oscar.vertices(P)))

true

In [ ]:
binomial(15,7)

In [ ]:
affine_hull(P_bar)

In [ ]:
affine_hull(P)

In [ ]:
print_constraints(P_bar)

In [ ]:
print_constraints(P)

#
***

# Compute the restriction of Q-Matroid to a given space
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 0])
QM = q_matroid_from_matrix(Mat)

Q-Matroid of rank 1 in 3-dim. vector-space over the Galois field with characteristic 2

In [366]:
space = matrix(GF(2),[1 0 0;0 1 0])
@time RQM = Restriction_Q_Matroid(QM,space)

  0.137905 seconds (234.83 k allocations: 15.831 MiB, 99.64% compilation time)


Embbeded q-Matroid of dim. 2 and rank 1 in 3-dim. vector-space over the Galois field with characteristic 2

In [367]:
RQM.em_bases

2-element Vector{Any}:
 [1   1   0]
 [1   0   0]

#
***

# Compute the spanning spaces of a q-matroid
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 x^2;0 1 x])
QM = q_matroid_from_matrix(Mat)
QM.bases

7-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]

In [243]:
Q_Matroid_Spanningspaces(QM)

8-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]
 [1 0 0; 0 1 0; 0 0 1]

# Compute the non spanning spaces of a q-matroid
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 x^2;0 1 x])
QM = q_matroid_from_matrix(Mat)
QM.bases

7-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]

In [245]:
Q_Matroid_Spanningspaces(QM)

8-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]
 [1 0 0; 0 1 0; 0 0 1]

In [246]:
Q_Matroid_Non_Spanningspaces(QM)

8-element Vector{Any}:
 [0   0   0]
 [1   0   0]
 [0   1   0]
 [1   1   0]
 [0   0   1]
 [1   0   1]
 [0   1   1]
 [1   1   1]

# Compute the open-spaces of a q-matroid
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[0 0 0])
QM = q_matroid_from_matrix(Mat)
QM.bases

1-element Vector{Any}:
 [0   0   0]

In [248]:
Q_Matroid_CircuitsV2(QM)

7-element Vector{Any}:
 [1   0   0]
 [0   1   0]
 [1   1   0]
 [0   0   1]
 [1   0   1]
 [0   1   1]
 [1   1   1]

In [249]:
Q_Matroid_Openspaces(QM)

15-element Vector{Any}:
 [1 0 0; 0 1 0; 0 0 1]
 [0 1 0; 0 0 1]
 [1 0 0; 0 0 1]
 [1 1 0; 0 0 1]
 [1 0 0; 0 1 0]
 [1 0 1; 0 1 0]
 [1 0 0; 0 1 1]
 [1 0 1; 0 1 1]
 [0   0   1]
 [0   1   0]
 [0   1   1]
 [1   0   0]
 [1   0   1]
 [1   1   0]
 [1   1   1]

# Checking the diamond property
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 0;0 1 x])
QM = q_matroid_from_matrix(Mat)
Indeps = Q_Matroid_Independentspaces(QM)
Deps = Q_Matroid_Dependentspaces(QM)

2-element Vector{Any}:
 [0 1 0; 0 0 1]
 [1 0 0; 0 1 0; 0 0 1]

In [252]:
space_1 = Indeps[1]
space_2 = Indeps[7]

[1   1   1]

In [253]:
#deleteat!(Indeps,1)
insert!(Deps,1,Indeps[5])

3-element Vector{Any}:
 [1   0   1]
 [0 1 0; 0 0 1]
 [1 0 0; 0 1 0; 0 0 1]

In [254]:
@time Diamond_prop(Indeps,Deps)

  0.575621 seconds (877.08 k allocations: 55.799 MiB, 99.78% compilation time)


false

In [255]:
all_subs = all_subspaces(base_ring(QM.groundspace),ncols(QM.bases[1]))
ds = diamond_list(all_subs);

In [256]:
Diamond_prop(Indeps,Deps,ds)

false

# Checking the encoded diamond property
***
***

In [ ]:
all = all_subspaces(GF(2),3)
Ext_F,x = finite_field(2,3,"x")
Mat = matrix(Ext_F,[1 0 0])
QM = q_matroid_from_matrix(Mat)
Indeps = Q_Matroid_Indepentspaces(QM)
Deps = Q_Matroid_Depentspaces(QM)

In [ ]:
ones_dict = OrderedDict([id-1=>elm for (id,elm) in enumerate(all) if rank(elm)==1])
encoded_all, q = sub_encoding(all,ones_dict,true)
En_Indeps = sub_encoding(Indeps,ones_dict)
En_Deps = sub_encoding(Deps,ones_dict)
en_all_diams = encoded_k_interval(encoded_all,q,2,encoded_all)

In [ ]:
insert!(En_Deps,1,En_Indeps[4])
#deleteat!(En_Indeps,2)

In [ ]:
@time En_Diamond_prop(En_Indeps,En_Deps,en_all_diams)

# 
***

# Enumeration Dimension 3 (rank 2)
***
***

In [ ]:
Ext_F,x = finite_field(2,2,"x")
A = matrix(Ext_F,[1 0; 0 1])
QM = q_matroid_from_matrix(A)

In [ ]:
indeps = Q_Matroid_Indepentspaces(QM)
deps = Q_Matroid_Depentspaces(QM)
indeps,deps

In [ ]:
@time triples = Dim3_q_matroid_DFS(QM)

In [310]:
all_subs = all_subspaces(QM.field,3)
for tri in triples
    uni = union(tri[1],tri[2])
    for space in all_subs
        if !(space in uni)
            println(space)
        end
    end
end

In [ ]:
true_list_alt = []
false_list_alt = []
for triple in triples
    answer = Are_q_matroid_dependentspaces(triple[2])
    if answer == true
        push!(true_list_alt,[length(triple[3]),length(triple[1])+length(triple[2]),triple[3],triple[1],triple[2]])
    else
        push!(false_list_alt,[length(triple[3]),length(triple[1])+length(triple[2]),triple[3],triple[2]])
    end
end
true_list_alt

In [ ]:
for elm in true_list_alt
    new_elm1 = unique(elm[5])
    println(length(elm[5]))
    println(length(new_elm1))
    println("------")
end

In [ ]:
false_list_alt

# Enumeration Dimension 4 (rank 2)
***
***

In [ ]:
Ext_F,x = finite_field(2,4,"x")
A = matrix(Ext_F,[1 0 x])
QM = q_matroid_from_matrix(A)

In [ ]:
indeps = Q_Matroid_Indepentspaces(QM)
deps = Q_Matroid_Depentspaces(QM)
indeps,deps

In [ ]:
@time triples = Dim4_q_matroid_DFS(QM)

In [ ]:
true_list_alt = []
false_list_alt = []
for triple in triples
    answer = Are_q_matroid_dependentspaces(triple[2])
    if answer == true
        push!(true_list_alt,[length(triple[3]),length(triple[1])+length(triple[2]),triple[3],triple[1],triple[2]])
    else
        push!(false_list_alt,[length(triple[3]),length(triple[1])+length(triple[2]),triple[3],triple[2]])
    end
end
true_list_alt

In [ ]:
for elm in true_list_alt
    println(intersect(elm[4],elm[5]))
end

# Enumeration Dimension 5 (rank 2)
***
***

In [ ]:
Ext_F,x = finite_field(2,4,"x")
A = matrix(Ext_F,[1 0 0 x])
QM = q_matroid_from_matrix(A)

In [ ]:
indeps = Q_Matroid_Indepentspaces(QM)
deps = Q_Matroid_Depentspaces(QM)
indeps,deps

In [ ]:
triples = Dim5_q_matroid_DFS(QM)

In [ ]:
true_list_alt = []
false_list_alt = []
for triple in triples
    answer = Are_q_matroid_dependentspaces(triple[2])
    if answer == true
        push!(true_list_alt,[length(triple[3]),length(triple[1])+length(triple[2]),triple[3],triple[1],triple[2]])
    else
        push!(false_list_alt,[length(triple[3]),length(triple[1])+length(triple[2]),triple[3],triple[2]])
    end
end
true_list_alt

In [ ]:
for elm in true_list_alt
    println(intersect(elm[4],elm[5]))
end

In [ ]:
groundspace = matrix_space(GF(2),5,5)(1)
nQM = Q_Matroid(groundspace,true_list_alt[1][3])

In [ ]:
Is_representable(nQM)

# Encoded Enumeration of rank-2 q-matroids
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
A = matrix(Ext_F,[1 0 x^2;0 1 x])
QM = q_matroid_from_matrix(A)

In [ ]:
indeps = Q_Matroid_Indepentspaces(QM)
deps = Q_Matroid_Depentspaces(QM)
indeps,deps

In [ ]:
@time triples = En_q_matroid_DFS(QM)

In [ ]:
all_ones = subspaces_fix_dim(GF(2),1,4)
ones_dict = OrderedDict([id=>elm for (id,elm) in enumerate(all_ones)])
new_triples = [[sub_decoding(elm[1],ones_dict),sub_decoding(elm[2],ones_dict),sub_decoding(elm[3],ones_dict)] for elm in triples]

In [ ]:
true_list_alt = []
false_list_alt = []
for triple in new_triples
    answer = Are_q_matroid_dependentspaces(triple[2])
    if answer == true
        push!(true_list_alt,[length(triple[3]),length(triple[1])+length(triple[2]),triple[3],triple[1],triple[2]])
    else
        push!(false_list_alt,[length(triple[3]),length(triple[1])+length(triple[2]),triple[3],triple[2]])
    end
end
true_list_alt

In [ ]:
for elm in true_list_alt
    println(intersect(elm[4],elm[5]))
end

In [ ]:
false_list_alt

# Optimized encoded Enumeration of rank-2 q-matroids
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
A = matrix(Ext_F,[1 0 x^2;0 1 x])
QM = q_matroid_from_matrix(A)

In [ ]:
@time triples = EN_q_matroid_DFS(QM)

In [ ]:
all_ones = subspaces_fix_dim(GF(2),1,4)
ones_dict = OrderedDict([id=>elm for (id,elm) in enumerate(all_ones)])
new_triples = [[sub_decoding(elm.indeps,ones_dict),sub_decoding(elm.deps,ones_dict),sub_decoding(elm.maxis,ones_dict)] for elm in triples]

In [ ]:
true_list_alt = []
false_list_alt = []
for triple in new_triples
    answer = Are_q_matroid_dependentspaces(triple[2])
    if answer == true
        push!(true_list_alt,[length(triple[3]),length(triple[1])+length(triple[2]),triple[3],triple[1],triple[2]])
    else
        push!(false_list_alt,[length(triple[3]),length(triple[1])+length(triple[2]),triple[3],triple[2]])
    end
end
true_list_alt

# Counting linear q-subclasses 
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
A = matrix(Ext_F,[1 0 0;0 1 0])
QM = q_matroid_from_matrix(A)
HPS = Q_Matroid_Hyperplanes(QM)

3-element Vector{Any}:
 [1 0 0; 0 0 1]
 [0 1 0; 0 0 1]
 [1 1 0; 0 0 1]

In [45]:
L = Liftig_linear_q_subclassesV3(QM,[],nothing,false)

(gfp_mat[[1 0 0 0; 0 0 1 0], [0 1 0 0; 0 0 1 0], [1 1 0 0; 0 0 1 0], [0 0 1 0; 0 0 0 1], [1 0 0 1; 0 0 1 0], [0 1 0 1; 0 0 1 0], [1 1 0 1; 0 0 1 0]], 1)

In [46]:
Are_q_matroid_hyperplanes(L[1],"Yes")

(true, "Non")

In [47]:
QM2 = q_matroid_from_hyperplanes(L[1])
length(QM2.bases)

28

In [19]:
HPS2 = Q_Matroid_Hyperplanes(QM2)

15-element Vector{Any}:
 [1   0   0   0]
 [0   1   0   0]
 [1   1   0   0]
 [0   0   1   0]
 [1   0   1   0]
 [0   1   1   0]
 [1   1   1   0]
 [0   0   0   1]
 [1   0   0   1]
 [0   1   0   1]
 [1   1   0   1]
 [0   0   1   1]
 [1   0   1   1]
 [0   1   1   1]
 [1   1   1   1]

In [20]:
Is_paving_q_matroid(QM2)

true

# Hyperplane Completion 
***
***

In [ ]:
Ext_F,x = finite_field(2,4,"x")
A = matrix(Ext_F,[1 0 0 x; 0 1 1 x^2])
QM = q_matroid_from_matrix(A)
HPS1 = Q_Matroid_Hyperplanes(QM)

7-element Vector{Any}:
 [1 0 0 0; 0 1 1 0]
 [0 1 0 0; 0 0 1 0]
 [1 0 1 0; 0 1 1 0]
 [0 1 1 0; 0 0 0 1]
 [1 0 0 1; 0 1 1 0]
 [0 1 0 1; 0 0 1 1]
 [1 0 1 1; 0 1 1 0]

In [3]:
bases31 = AbstractVector{Any}([])
two_spaces = subspaces_fix_dim(GF(2),2,4)
A = matrix(GF(2),[1 0 0 0;0 1 0 0])
B = matrix(GF(2),[0 0 1 0;0 0 0 1])
C = matrix(GF(2),[1 0 0 1;0 1 1 0])
D = matrix(GF(2),[1 0 1 0;0 1 1 1])
l = [A,B,C,D]
for space in two_spaces
    if !(space in l)
        push!(bases31,space)
    end
end
qm31 = Q_Matroid(matrix(GF(2),[1 0 0 0;0 1 0 0;0 0 1 0;0 0 0 1]),bases31)
HPS31 = Q_Matroid_Hyperplanes(qm31)

7-element Vector{Any}:
 [1   1   1   0]
 [0   1   0   1]
 [1   0   1   1]
 [1 0 0 0; 0 1 0 0]
 [0 0 1 0; 0 0 0 1]
 [1 0 1 0; 0 1 1 1]
 [1 0 0 1; 0 1 1 0]

In [158]:
length(qm31.bases)

31

In [159]:
function Hyperplane_completion(spaces,Low=true,count_bound=15)

    field = base_ring(spaces[1])
    dim = ncols(spaces[1])
    count = 0

    # Completion of spaces w.r.t. the Hyperplane-Axiom (H3)
    if Are_q_matroid_hyperplanes(spaces)
        return spaces,count
    else
        one_spaces = subspaces_fix_dim(field,1,dim)
        Completion = spaces
        are_no_hyperplanes = true
        checker = true

        while are_no_hyperplanes
            for combi in combinations(Completion,2)
                inters = inters_vsV3(combi[1],combi[2])
                ones_A = dim_one_subs(combi[1])
                ones_B = dim_one_subs(combi[2])
                leftoverones = [x for x in one_spaces if !(x in ones_A) && !(x in ones_B)]
                for z in leftoverones
                    sum = sum_vsV2(inters, z)
                    if !(sum in Completion)
                        push!(Completion,sum)
                    end
                end
            end
            Completion = unique(Completion)

            # Check for subspace relation and always keep the higher or lower dimensional space if there is one
            Completion_list = [[x, rank(x), subspace_set(x)] for  x in Completion]
            Remove_list = []
            for combi in combinations(Completion_list,2)
                if Low
                    if issubset(combi[1][3],combi[2][3]) || issubset(combi[2][3],combi[1][3])
                        if combi[1][2] >= combi[2][2]
                            push!(Remove_list,combi[2][1])
                        elseif combi[2][2] > combi[1][2]
                            push!(Remove_list,combi[1][1])
                        end
                    end
                elseif Low == false
                    if issubset(combi[1][3],combi[2][3]) || issubset(combi[2][3],combi[1][3])
                        if combi[1][2] <= combi[2][2]
                            push!(Remove_list,combi[2][1])
                        elseif combi[2][2] < combi[1][2]
                            push!(Remove_list,combi[1][1])
                        end
                    end
                end
            end

            Completion = [x[1] for x in Completion_list if !(x[1] in Remove_list)]
            count += 1 

            if Are_q_matroid_hyperplanes(Completion)
                checker = true
                are_no_hyperplanes = false
            elseif count >= count_bound
                checker = false
                are_no_hyperplanes = false
            end

        end

        if checker
            return Completion,count
        else
            message = "Completion not possible"
            return message
        end
    end
    
end

Hyperplane_completion (generic function with 3 methods)

In [160]:
f=GF(2)
spaces = AbstractVector{Any}(HPS31[4:7])
HPS = Hyperplane_completion(spaces,true)

(gfp_mat[[1 0 0 0; 0 1 0 0], [0 0 1 0; 0 0 0 1], [1 0 1 0; 0 1 1 1], [1 0 0 1; 0 1 1 0], [1 1 1 0], [0 1 0 1], [1 0 1 1]], 1)

In [162]:
Set(HPS31) == Set(HPS[1])

true

In [163]:
Are_q_matroid_hyperplanes(HPS[1])

true

In [164]:
QM2 = q_matroid_from_hyperplanes(HPS[1])
length(QM2.bases)

31

In [135]:
Is_paving_q_matroid(QM2)

true

# 
***

# Checking if two q-matroids are q-quotients and a collection is q-concordant
***
***

In [ ]:
Ext_F1,x = finite_field(2,3,"x")
Ext_F2,y = finite_field(2,3,"y")
Ext_F3,z = finite_field(2,3,"z")
A1 = matrix(Ext_F1,[0 0 1])
A2 = matrix(Ext_F2,[0 1 0; 0 0 1])
A3 = matrix(Ext_F2,[1 0 0; 0 1 0; 0 0 1])
QM1 = q_matroid_from_matrix(A1)
QM2 = q_matroid_from_matrix(A2)
QM3 = q_matroid_from_matrix(A3)

Q-Matroid of rank 3 in 3-dim. vector-space over the Galois field with characteristic 2

In [267]:
Q_Matroid_CircuitsV2(QM1)

3-element Vector{Any}:
 [1   0   0]
 [0   1   0]
 [1   1   0]

In [268]:
Q_Matroid_CircuitsV2(QM2)

1-element Vector{Any}:
 [1   0   0]

In [269]:
Q_Matroid_CircuitsV2(QM3)

Any[]

In [270]:
Are_q_quotients(QM1,QM2)

true

In [271]:
Is_q_concordant_collec([QM1,QM2,QM3])

true

# Compute the Q-Higgs-Lift off two q-matroids
***
***

In [ ]:
Ext_F1,x = finite_field(2,3,"x")
Ext_F2,y = finite_field(2,3,"y")
Ext_F3,z = finite_field(2,3,"z")
A1 = matrix(Ext_F1,[1 x x^2])
A2 = matrix(Ext_F2,[0 1 0; 0 0 1])
A3 = matrix(Ext_F2,[1 0 0; 0 1 0; 0 0 1])
QM1 = q_matroid_from_matrix(A1)
QM2 = q_matroid_from_matrix(A2)
QM3 = q_matroid_from_matrix(A3)

Q-Matroid of rank 3 in 3-dim. vector-space over the Galois field with characteristic 2

In [297]:
newQM = Q_Higgs_lift(QM1,QM3)

Q-Matroid of rank 2 in 3-dim. vector-space over the Galois field with characteristic 2

In [298]:
newQM.bases

7-element Vector{Any}:
 [1 0 0; 0 1 0]
 [1 0 0; 0 0 1]
 [1 0 0; 0 1 1]
 [0 1 0; 0 0 1]
 [1 0 1; 0 1 0]
 [1 1 0; 0 0 1]
 [1 0 1; 0 1 1]

# Compute the automorphism group of a q-matroid
***
***

In [ ]:
Ext_F,x = finite_field(2,3,"x")
B = matrix(Ext_F,[1 0 0;0 1 0])
QM_B = q_matroid_from_matrix(B)

Q-Matroid of rank 2 in 3-dim. vector-space over the Galois field with characteristic 2

In [27]:
G = Oscar.general_linear_group(2, GF(2))
E = collect(Oscar.elements(G))
typeof(E[1])

MatrixGroupElem{gfp_elem, gfp_mat}

In [30]:
GB = Q_Matroid_Aut(QM_B)

Matrix group of degree 3 over Galois field with characteristic 2

In [31]:
Oscar.describe(GB)

"S4"

# 
***

# Database
***
***

## Connect to database

In [ ]:
db = SQLite.DB("./q_matroids_db")
con = DBInterface

In [ ]:
SQLite.tables(db)

## Insert entries

In [ ]:
# Ext_F1,x = finite_field(2,2,"x")
# Ext_F2,y = finite_field(2,3,"y")
# matrices_list = [matrix(Ext_F1,[0 0]),matrix(Ext_F1,[1 0]),matrix(Ext_F1,[0 1]),matrix(Ext_F1,[1 1]),matrix(Ext_F1,[1 x]),matrix(Ext_F1,[1 0;0 1]),
#                  matrix(Ext_F2,[0 0 0]),matrix(Ext_F2,[1 0 0]),matrix(Ext_F2,[0 1 0]),matrix(Ext_F2,[0 0 1]),matrix(Ext_F2,[1 0 y]),
#                  matrix(Ext_F2,[1 y 0]),matrix(Ext_F2,[y 1 0]),matrix(Ext_F2,[1 y y^2]),matrix(Ext_F2,[1 0 0;0 1 0]),matrix(Ext_F2,[1 0 0;0 0 1]),
#                  matrix(Ext_F2,[0 1 0;0 0 1]),matrix(Ext_F2,[1 0 0;0 1 y]),matrix(Ext_F2,[1 y 0;0 0 1]),matrix(Ext_F2,[1 0 y;0 1 0]),matrix(Ext_F2,[1 0 y^2;0 1 y]),
#                  matrix(Ext_F2,[1 0 0;0 1 0;0 0 1])];

In [30]:
# for mat in matrices_list
#     QM = q_matroid_from_matrix(mat)
#     Add_entry(QM,db,con)
# end

## Query database for entries

In [ ]:
DataFrame(con.execute(db, "SELECT Base_spaces from q_Matroids WHERE Dim=2"))

## Delete entries

In [ ]:
con.execute(db, "DELETE from q_Matroids WHERE Encoding=1010101")

# 
***

In [ ]:
Ext_F1,x = finite_field(2,2,"x")
Ext_F2,y = finite_field(2,3,"y")
A = matrix(Ext_F1,[1 0 0;0 1 x])
B = matrix(Ext_F2,[1 0 0 y; 0 1 0 y^2])
QM1 = q_matroid_from_matrix(A)
QM2 = q_matroid_from_matrix(B)

In [ ]:
A = transpose(QQ[1 0 2 0 1 0; 0 1 2 0 0 1; 2 0 1 0 1 0; 2 0 0 1 0 1; 0 2 1 0 0 1; 0 2 0 1 1 0; 1 0 0 2 0 1; 0 1 0 2 1 0; 1 0 1 0 2 0; 0 1 0 1 2 0; 0 1 1 0 0 2; 1 0 0 1 0 2])

b = transpose(QQ[1 0 1 1 0 1])

Oscar.solve(A,b)